# Chi-square tests
Goodness of fit and the test of independence, by hand and with scipy, then the Titanic case study.
The 60-person table comes from `570-choosing-a-hypothesis-test/data/people.csv`; the counts are typed in here.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

## 1. Goodness of fit: age groups against census shares 25% / 55% / 20%

In [ ]:
observed = np.array([20, 26, 14])          # child, adult, elderly
shares = np.array([0.25, 0.55, 0.20])
expected = observed.sum() * shares          # n times each claimed share
parts = (observed - expected) ** 2 / expected
print('expected', expected, ' parts', parts.round(3))
chi2 = parts.sum()
df = len(observed) - 1
print(f'chi2 = {chi2:.3f}, df = {df}, p = {stats.chi2.sf(chi2, df):.3f}')
print(stats.chisquare(observed, expected))

## 2. Two categories: chi-square equals z squared

In [ ]:
gof = stats.chisquare([26, 34], [30, 30])
z = (26 / 60 - 0.5) / np.sqrt(0.25 / 60)
print('chi2 =', round(gof.statistic, 4), ' z^2 =', round(z ** 2, 4), ' p =', round(gof.pvalue, 4))

## 3. Critical values of the chi-square distribution

In [ ]:
for df in [1, 2, 3]:
    print(df, 'df: 5% critical value', round(stats.chi2.ppf(0.95, df), 2))

## 4. Test of independence by hand: gender by age group

In [ ]:
table = pd.DataFrame([[12, 12, 10], [8, 14, 4]], index=['female', 'male'],
                     columns=['child', 'adult', 'elderly'])
n = table.values.sum()
# expected = row total * column total / n, for every cell at once
expected = np.outer(table.sum(axis=1), table.sum(axis=0)) / n
print(expected.round(2))
parts = (table.values - expected) ** 2 / expected
print(parts.round(3))
chi2 = parts.sum()
df = (table.shape[0] - 1) * (table.shape[1] - 1)
print(f'chi2 = {chi2:.2f}, df = {df}, p = {stats.chi2.sf(chi2, df):.2f}')

In [ ]:
result = stats.chi2_contingency(table)
print(result.statistic, result.pvalue, result.dof)

## 5. Titanic: sex and survival

In [ ]:
titanic = pd.read_csv('data/titanic_train.csv')
sex = pd.crosstab(titanic['Sex'], titanic['Survived'])
print(sex)
plain = stats.chi2_contingency(sex, correction=False)
print('expected:\n', plain.expected_freq.round(1))
print('plain chi2', round(plain.statistic, 1), ' p', plain.pvalue)
yates = stats.chi2_contingency(sex)       # Yates' correction is the default for 2 x 2
print('Yates chi2', round(yates.statistic, 1))
print("Cramer's V", round(np.sqrt(plain.statistic / len(titanic)), 2))

## 6. Titanic: class and survival

In [ ]:
cls = pd.crosstab(titanic['Pclass'], titanic['Survived'])
res = stats.chi2_contingency(cls)
print(cls)
print('expected:\n', res.expected_freq.round(1))
print(f'chi2 = {res.statistic:.1f}, df = {res.dof}, p = {res.pvalue:.1e}')
print('survival rate by class', (cls[1] / cls.sum(axis=1)).round(2).tolist())
print("Cramer's V", round(stats.contingency.association(cls, method='cramer'), 2))